In [ ]:
# Cell 1: Environment Setup & Library Import
import sys
from pathlib import Path

# Add project root directory to path
sys.path.append(str(Path.cwd().parent))

# Import all modular library functions
from src import (
    load_data, 
    encode_categorical_features, 
    engineer_risk_features, 
    prepare_feature_matrix, 
    train_xgboost_model, 
    generate_risk_scores)

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Cell 2: Load Processed Data & Run Steps 1 & 2 Pipelines
PROCESSED_PATH = "../data/proc_fault_data.csv"

df = load_data(PROCESSED_PATH)
df_encoded = encode_categorical_features(df, cat_cols=['Weather Condition', 'Maintenance Status'])
df_featured = engineer_risk_features(df_encoded)

# Prepare Feature Matrix (X) and Target Vector (y)
X, y = prepare_feature_matrix(df_featured, target_col='is_critical', exclude_cols=['Fault ID', 'Component Health', 'Asset ID', 'Fault Location (Latitude, Longitude)', 'Fault Type'])
print(f"Training Matrix Shape: X={X.shape}, y={y.shape}")

In [ ]:
# Cell 3: Train XGBoost Classifier & Evaluate
model, scaler, X_test, y_test, auc_score, report, conf_matrix = train_xgboost_model(X, y)

print(f"Model ROC-AUC Score: {auc_score:.4f}\n")
print("Classification Report:\n", report)

# Display Confusion Matrix
print("Confusion Matrix:\n", conf_matrix)

# Optional: Plot Confusion Matrix visually
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Normal (0)', 'Predicted Critical (1)'],
            yticklabels=['Actual Normal (0)', 'Actual Critical (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('XGBoost Risk Classification - Confusion Matrix')
plt.show()

In [ ]:
# Cell 4: Generate 0–100 Risk Scores for All Assets
risk_scores = generate_risk_scores(model, scaler, X)
df_featured['risk_score_0_100'] = risk_scores

print("\nSample Asset Risk Scores (0–100):")
display_cols = [col for col in ['Fault ID', 'Voltage (V)', 'Wind Speed (km/h)', 'Temperature (°C)',  'risk_score_0_100'] if col in df_featured.columns]
print(df_featured[display_cols].head(10))

